In [ ]:
# ═══════════════════════════════════════════════════════════
#  EEG_10 — Hypergraph Neural Networks per Imagined Speech
#  Modelli: HGNN_2L (statico), HGNN_2L_DYN (dinamico), HGNN_ATT_2L (attention)
#  Riferimenti: Feng et al. 2019 (HGNN), Li et al. 2025 (DHSLP), Chien et al. 2022 (AllSet)
# ═══════════════════════════════════════════════════════════
USE_CLUSTERS   = True
CLUSTER_SCHEME = "concr4"  # "ward4" | "concr4" | "sem5" | "pos4"
# ───────────────────────────────────────────────────────────
K_HYPER        = 6     # vicini per iperspigolo (size iperspigolo = K+1)
HIDDEN_DIM     = 64    # hidden dim HypergraphConv
NODE_EMB_DIM   = 64    # output Temporal Encoder
HGNN_HEADS     = 4     # attention heads (HGNN_ATT)
DROPOUT        = 0.5
# ═══════════════════════════════════════════════════════════

In [ ]:
import os, math
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import json as _json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

# PyTorch Geometric
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import HypergraphConv, global_mean_pool

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)
print("Python:", __import__("sys").version.split()[0])
print("torch:", torch.__version__)
import torch_geometric; print("torch_geometric:", torch_geometric.__version__)

In [ ]:
# ============================================================
# CONFIGURAZIONE PATHS E IPERPARAMETRI TRAINING
# ============================================================

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)

META_CSV  = project_root / "data" / "interim" / "eeg_metadata.csv"
ELOC_PATH = project_root / "src" / "io" / "ebneuro.locs"
interim_dir = project_root / "data" / "interim"

N_CHANS = 59
N_TIMES = 384
SFREQ   = 256

BATCH_SIZE   = 32
MAX_EPOCHS   = 100
PATIENCE     = 20
LR           = 1e-3
WEIGHT_DECAY = 1e-4

SUBJ_TRAIN = list(range(50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

RESUME = True
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"project_root: {project_root}")
print("Config OK")

In [ ]:
# ============================================================
# METADATA + CLUSTER MAPPING
# Identico a EEG_08/09
# ============================================================

import sys
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

meta = pd.read_csv(META_CSV)
_initial_len = len(meta)
meta = meta[~(
    (meta["path_h5"].str.contains("08_05.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_01.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_03.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_04.h5") & (meta["epoch_idx"] >= 34))
)]
if len(meta) < _initial_len:
    print(f"Rimosse {_initial_len - len(meta)} righe corrotte.")
meta["subject_id"] = meta["subject_id"].astype(str).str.zfill(2)


def read_eloc_names(path):
    names = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]


ch_names_61 = read_eloc_names(ELOC_PATH)
EXCLUDE    = {"A1", "A2"}
keep_idx   = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
keep_names = [ch_names_61[i] for i in keep_idx]
assert len(keep_idx) == N_CHANS, f"Attesi {N_CHANS} canali, trovati {len(keep_idx)}"

_scheme = CLUSTER_SCHEME if USE_CLUSTERS else "raw110"
labelid2cluster, N_CLASSES, cluster_names = load_label_scheme(_scheme, interim_dir)

RESULTS_CSV = project_root / "data" / "interim" / f"eeg10_hgnn_{N_CLASSES}_results.csv"

print(f"Schema: {CLUSTER_SCHEME if USE_CLUSTERS else '110 parole'}")
print(f"Classi: {N_CLASSES}  |  Chance level: {100/N_CLASSES:.1f}%")
print(f"Canali: {len(keep_names)} — {keep_names[:5]}...")
print(f"Totale record: {len(meta)}")

In [ ]:
# ============================================================
# MATRICE PCC + COSTRUZIONE HYPEREDGE_INDEX
#
# Differenza rispetto a EEG_08/09:
#   - EEG_08/09: PCC k-NN → edge_index (coppie di nodi)
#   - EEG_10:    PCC k-NN → hyperedge_index (iperspigoli di ordine k+1)
#
# Per ogni nodo i: e_i = {i} ∪ {top-k nodi più correlati per |PCC|}
# Risultato: N=59 iperspigoli, ciascuno di dimensione k+1
# Ispirazione: Li et al. 2025 (DHSLP) — distanza-based hyperedge structure
# ============================================================


def compute_pcc_matrix(meta_df, keep_idx, subj_train_ids,
                       n_samples=1000, seed=42):
    """
    Calcola la matrice di Pearson Correlation Coefficient (|PCC|) media
    tra tutti i 59 canali su trial di training campionati.
    Restituisce matrice [59, 59].
    """
    rng = np.random.RandomState(seed)
    train_records = meta_df[meta_df["subject_id"].isin(
        [str(i).zfill(2) for i in subj_train_ids]
    )][["path_h5", "epoch_idx"]]
    n = min(n_samples, len(train_records))
    sampled = train_records.iloc[rng.choice(len(train_records), n, replace=False)]

    paths_map = defaultdict(list)
    for _, row in sampled.iterrows():
        paths_map[row["path_h5"]].append(int(row["epoch_idx"]))

    buf = []
    print(f"Calcolo PCC su {n} trial da {len(paths_map)} file H5...")
    for p, epoch_idxs in tqdm(paths_map.items(), desc="PCC H5", leave=False):
        with h5py.File(p, "r") as f:
            for e_idx in epoch_idxs:
                buf.append(f["data"][e_idx][keep_idx, :].astype(np.float32))

    pcc_sum = np.zeros((len(keep_idx), len(keep_idx)), dtype=np.float64)
    for epoch_data in buf:
        pcc = np.corrcoef(epoch_data)
        pcc_sum += np.abs(pcc)
    pcc_mean = pcc_sum / len(buf)
    np.fill_diagonal(pcc_mean, 0.0)
    print(f"PCC calcolato su {len(buf)} trial | range: [{pcc_mean.min():.3f}, {pcc_mean.max():.3f}]")
    return pcc_mean


def pcc_to_hyperedge_index(pcc_matrix, k=6):
    """
    Converte matrice PCC [N×N] in hyperedge_index formato PyG.

    Per ogni nodo i crea iperspigolo e_i = {i} ∪ {top-k più correlati}.
    → N iperspigoli di dimensione k+1 ciascuno.

    Args:
        pcc_matrix: np.ndarray [N, N] — |PCC| medio tra canali
        k: int — numero di vicini per iperspigolo

    Returns:
        hyperedge_index: torch.LongTensor [2, N*(k+1)]
            row 0: vertex indices (nodi membri dell'iperspigolo)
            row 1: hyperedge indices (0…N-1)
        N_HYPER: int — numero totale di iperspigoli (= N)
    """
    N = pcc_matrix.shape[0]
    vertex_list, edge_list = [], []

    for i in range(N):
        row = pcc_matrix[i].copy()
        row[i] = -1.0  # escludi autoconnessione
        top_k = np.argsort(row)[-k:]  # k canali più correlati
        members = [i] + top_k.tolist()
        for v in members:
            vertex_list.append(v)
            edge_list.append(i)  # iperspigolo i-esimo

    hyperedge_index = torch.tensor([vertex_list, edge_list], dtype=torch.long)
    print(f"hyperedge_index: {N} iperspigoli, {hyperedge_index.shape[1]} entries totali")
    print(f"  Ogni iperspigolo ha esattamente {k+1} nodi membri")
    return hyperedge_index, N


# Calcola PCC e costruisci hyperedge_index
pcc_matrix   = compute_pcc_matrix(meta, keep_idx, SUBJ_TRAIN)
hyperedge_index, N_HYPER = pcc_to_hyperedge_index(pcc_matrix, k=K_HYPER)
print(f"
N_HYPER = {N_HYPER} iperspigoli (uno per canale EEG)")
print(f"Incidence matrix size: {N_CHANS} × {N_HYPER}")

In [ ]:
# ============================================================
# HYPERGRAPHDATA + DATASET
# ============================================================

class HypergraphData(Data):
    """
    Subclasse Data PyG con supporto corretto per hyperedge_index nel batching.

    PyG incrementa automaticamente gli indici tensoriali durante il batching.
    Per hyperedge_index [2, E]:
      - row 0 (vertex indices):    incrementa di num_nodes per grafo
      - row 1 (hyperedge indices): incrementa di num_hyperedges per grafo

    Senza questo override, PyG incrementerebbe entrambe le righe di num_nodes,
    causando indici hyperedge errati nel batch.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def __inc__(self, key, value, *args, **kwargs):
        if key == "hyperedge_index":
            return torch.tensor([[self.num_nodes], [int(self.num_hyperedges)]])
        return super().__inc__(key, value, *args, **kwargs)


class EEGHypergraphDataset(Dataset):
    """
    Dataset PyG per modelli ipergraph.
    Ogni trial EEG → HypergraphData con:
      - x: [59, 384]  segnale normalizzato per canale
      - hyperedge_index: [2, N*(K+1)]  iperspigoli statici da PCC
      - num_hyperedges: int  (per batching corretto in HypergraphData)
      - y: label cluster (int)
    """

    def __init__(self, records, keep_idx, labelid2cluster,
                 hyperedge_index, n_hyper, mean=None, std=None):
        self.records          = records
        self.keep_idx         = keep_idx
        self.labelid2cluster  = labelid2cluster
        self.hyperedge_index  = hyperedge_index
        self.n_hyper          = n_hyper
        self.mean             = mean
        self.std              = std
        self.file_cache       = {}
        if mean is None:
            self._compute_stats()

    def _compute_stats(self, seed=42):
        rng   = np.random.RandomState(seed)
        n     = min(500, len(self.records))
        idxs  = rng.choice(len(self.records), n, replace=False)
        paths_map = defaultdict(list)
        for idx in idxs:
            r = self.records[idx]
            paths_map[r["path_h5"]].append(int(r["epoch_idx"]))
        buf = []
        print(f"Calcolo stats su {n} campioni...")
        for path, epoch_idxs in tqdm(paths_map.items(), desc="Stats H5", leave=False):
            with h5py.File(path, "r") as f:
                for e_idx in epoch_idxs:
                    buf.append(f["data"][e_idx][self.keep_idx, :].astype(np.float32))
        buf = np.stack(buf)
        self.mean = torch.tensor(buf.mean(axis=(0, 2), keepdims=True).squeeze(0), dtype=torch.float32)
        self.std  = torch.tensor(buf.std(axis=(0, 2),  keepdims=True).squeeze(0).clip(1e-6), dtype=torch.float32)
        print(f"Stats OK | mean: [{self.mean.min():.3f}, {self.mean.max():.3f}]")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r    = self.records[idx]
        path = r["path_h5"]
        if path not in self.file_cache:
            self.file_cache[path] = h5py.File(path, "r")
        x = self.file_cache[path]["data"][int(r["epoch_idx"])][self.keep_idx, :].astype(np.float32)
        x = (torch.tensor(x, dtype=torch.float32) - self.mean) / self.std
        label = self.labelid2cluster[int(r["label_idx"])]
        return HypergraphData(
            x              = x,
            hyperedge_index= self.hyperedge_index.clone(),
            num_hyperedges = torch.tensor(self.n_hyper, dtype=torch.long),
            y              = torch.tensor(label, dtype=torch.long),
        )

    def __del__(self):
        for f in self.file_cache.values():
            try: f.close()
            except: pass


def make_hypergraph_splits(meta_df, labelid2cluster, subj_train, subj_val, subj_test,
                           hyperedge_index, n_hyper):
    def to_records(df_sub):
        return df_sub[["path_h5", "epoch_idx", "label_idx", "subject_id"]].to_dict("records")

    train_ids = [str(i).zfill(2) for i in subj_train]
    val_ids   = [str(i).zfill(2) for i in subj_val]
    test_ids  = [str(i).zfill(2) for i in subj_test]

    df_tr = meta_df[meta_df["subject_id"].isin(train_ids)]
    df_va = meta_df[meta_df["subject_id"].isin(val_ids)]
    df_te = meta_df[meta_df["subject_id"].isin(test_ids)]

    ds_tr = EEGHypergraphDataset(to_records(df_tr), keep_idx, labelid2cluster,
                                  hyperedge_index, n_hyper)
    ds_va = EEGHypergraphDataset(to_records(df_va), keep_idx, labelid2cluster,
                                  hyperedge_index, n_hyper,
                                  mean=ds_tr.mean, std=ds_tr.std)
    ds_te = EEGHypergraphDataset(to_records(df_te), keep_idx, labelid2cluster,
                                  hyperedge_index, n_hyper,
                                  mean=ds_tr.mean, std=ds_tr.std)

    print(f"Split: train={len(ds_tr)}, val={len(ds_va)}, test={len(ds_te)}")
    return ds_tr, ds_va, ds_te


print("HypergraphData + Dataset pronti")

In [ ]:
# ============================================================
# MODELLI HYPERGRAPH
#
# 1. HGNN_2L      — static hyperedges (PCC k-NN)
#                   Feng et al. 2019 — HGNN (AAAI)
#
# 2. HGNN_2L_DYN  — layer 1: static, layer 2: dynamic
#                   Struttura aggiornata nel feature space
#                   Ispirazione: Li et al. 2025 — DHSLP
#
# 3. HGNN_ATT_2L  — static + attention sugli iperspigoli
#                   use_attention=True in HypergraphConv
#                   Ispirazione: Chien et al. 2022 — AllSet
#
# TemporalEncoder — IDENTICO a EEG_08/09 (1D CNN per canale)
# ============================================================


# ── Temporal Encoder (invariato da EEG_08) ──────────────────
class TemporalEncoder(nn.Module):
    """
    1D CNN per-nodo con pesi condivisi tra tutti i canali.
    Input:  (N_nodes_total, N_TIMES)
    Output: (N_nodes_total, out_dim)
    """
    def __init__(self, n_times=N_TIMES, out_dim=NODE_EMB_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(32), nn.ELU(),
            nn.Conv1d(32, 64, kernel_size=10, stride=2, padding=5),
            nn.BatchNorm1d(64), nn.ELU(),
            nn.AdaptiveAvgPool1d(4),
            nn.Flatten(),
            nn.Linear(64 * 4, out_dim),
            nn.ELU(),
        )

    def forward(self, x):
        return self.net(x.unsqueeze(1))   # (N, 1, T) → (N, out_dim)


# ── Costruzione dinamica degli iperspigoli ──────────────────
def build_dynamic_hyperedge_index(x, batch, k=6):
    """
    Costruisce hyperedge_index dinamicamente dal feature space corrente.
    Ispirazione diretta: Li et al. 2025 (DHSLP) — distanza-based hyperedge.

    Per ogni nodo i in ogni grafo del batch:
        e_i = {i} ∪ {top-k nodi con cosine similarity più alta}

    Args:
        x:     [N_total, D]  — node features post primo layer
        batch: [N_total]     — batch assignment PyG
        k:     int           — numero di vicini

    Returns:
        hyperedge_index: [2, E] LongTensor (già offsettato per batch)
    """
    num_graphs   = int(batch.max().item()) + 1
    vertex_list, edge_list = [], []
    edge_offset = 0
    node_offset = 0

    for g in range(num_graphs):
        mask = (batch == g)
        x_g  = x[mask]   # [N_g, D]
        N_g  = x_g.shape[0]

        # Cosine similarity [N_g, N_g]
        x_norm = F.normalize(x_g.detach(), p=2, dim=1)
        sim    = x_norm @ x_norm.T

        # Top-k più simili per ogni nodo (include se stesso)
        _, topk = torch.topk(sim, k=min(k + 1, N_g), dim=1)  # [N_g, k+1]

        for i in range(N_g):
            for j in topk[i].tolist():
                vertex_list.append(node_offset + j)
                edge_list.append(edge_offset + i)

        node_offset += N_g
        edge_offset += N_g  # N_g iperspigoli per grafo

    return torch.tensor([vertex_list, edge_list], dtype=torch.long, device=x.device)


# ── HGNN_2L — iperspigoli statici ────────────────────────────
class HGNN_2L(nn.Module):
    """
    2 layer HypergraphConv con iperspigoli statici (PCC k-NN).
    Baseline hypergraph — confronto diretto con GAT_2L (EEG_09).
    Ref: Feng et al. 2019 — HGNN (AAAI)
    """
    def __init__(self, n_classes=N_CLASSES, n_times=N_TIMES,
                 node_emb_dim=NODE_EMB_DIM, hidden=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.temporal_enc = TemporalEncoder(n_times, node_emb_dim)
        self.conv1 = HypergraphConv(node_emb_dim, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = HypergraphConv(hidden, hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.drop  = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, data):
        x  = data.x
        he = data.hyperedge_index

        x = self.temporal_enc(x)

        x = F.elu(self.bn1(self.conv1(x, he)))
        x = self.drop(x)
        x = F.elu(self.bn2(self.conv2(x, he)))

        x = global_mean_pool(x, data.batch)
        return self.classifier(x)


# ── HGNN_2L_DYN — iperspigoli dinamici al layer 2 ────────────
class HGNN_2L_DYN(nn.Module):
    """
    Layer 1: HypergraphConv con struttura statica (PCC).
    Layer 2: HypergraphConv con struttura dinamica (feature cosine sim).
    Il secondo layer vede relazioni aggiornate al feature space corrente.
    Ispirazione: Li et al. 2025 — DHSLP (dynamic structure learning).
    """
    def __init__(self, n_classes=N_CLASSES, n_times=N_TIMES,
                 node_emb_dim=NODE_EMB_DIM, hidden=HIDDEN_DIM,
                 k=K_HYPER, dropout=DROPOUT):
        super().__init__()
        self.k = k
        self.temporal_enc = TemporalEncoder(n_times, node_emb_dim)
        self.conv1 = HypergraphConv(node_emb_dim, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = HypergraphConv(hidden, hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.drop  = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, data):
        x        = data.x
        he_static = data.hyperedge_index

        x = self.temporal_enc(x)

        # Layer 1: struttura statica (PCC k-NN)
        x = F.elu(self.bn1(self.conv1(x, he_static)))
        x = self.drop(x)

        # Aggiorna struttura ipergraph nel feature space corrente
        he_dyn = build_dynamic_hyperedge_index(x, data.batch, k=self.k)

        # Layer 2: struttura dinamica (cosine sim features)
        x = F.elu(self.bn2(self.conv2(x, he_dyn)))

        x = global_mean_pool(x, data.batch)
        return self.classifier(x)


# ── HGNN_ATT_2L — attention sugli iperspigoli ────────────────
class HGNN_ATT_2L(nn.Module):
    """
    2 layer HypergraphConv con attention (use_attention=True).
    Il meccanismo di attenzione pesa i contributi dei nodi
    dentro ogni iperspigolo — simile a GAT ma su hyperedge.
    Ispirazione: Chien et al. 2022 — AllSet (ICLR).
    """
    def __init__(self, n_classes=N_CLASSES, n_times=N_TIMES,
                 node_emb_dim=NODE_EMB_DIM, hidden=HIDDEN_DIM,
                 heads=HGNN_HEADS, dropout=DROPOUT):
        super().__init__()
        self.temporal_enc = TemporalEncoder(n_times, node_emb_dim)
        self.conv1 = HypergraphConv(node_emb_dim, hidden,
                                    use_attention=True, heads=heads,
                                    concat=False, dropout=dropout)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = HypergraphConv(hidden, hidden,
                                    use_attention=True, heads=heads,
                                    concat=False, dropout=dropout)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.drop  = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, data):
        x  = data.x
        he = data.hyperedge_index

        x = self.temporal_enc(x)

        x = F.elu(self.bn1(self.conv1(x, he)))
        x = self.drop(x)
        x = F.elu(self.bn2(self.conv2(x, he)))

        x = global_mean_pool(x, data.batch)
        return self.classifier(x)


# Registro modelli
HGNN_MODELS = {
    "HGNN_2L":     {"cls": HGNN_2L,     "desc": "Static hyperedges (PCC k-NN)"},
    "HGNN_2L_DYN": {"cls": HGNN_2L_DYN, "desc": "Dynamic hyperedges (layer 2)"},
    "HGNN_ATT_2L": {"cls": HGNN_ATT_2L, "desc": "Static + attention on hyperedges"},
}

total_params = sum(p.numel() for p in HGNN_2L().parameters())
print(f"Parametri HGNN_2L: {total_params:,}")
print(f"Modelli da trainare: {list(HGNN_MODELS.keys())}")

In [ ]:
# ============================================================
# CLASS WEIGHTS + SPLIT DATASET
# ============================================================

ds_train, ds_val, ds_test = make_hypergraph_splits(
    meta, labelid2cluster,
    SUBJ_TRAIN, SUBJ_VAL, SUBJ_TEST,
    hyperedge_index, N_HYPER
)

# Class weights inversi alla frequenza (anti class-collapse)
all_labels   = [labelid2cluster[int(r["label_idx"])] for r in ds_train.records]
counts       = torch.bincount(torch.tensor(all_labels), minlength=N_CLASSES).float()
class_weights = (1.0 / counts.clamp(min=1))
class_weights = (class_weights / class_weights.sum() * N_CLASSES).to(device)

print("
Class weights (training set):")
for i, (c, w) in enumerate(zip(cluster_names, class_weights.cpu())):
    print(f"  classe {i} ({c}): {int(counts[i])} trial → weight={w:.3f}")

In [ ]:
# ============================================================
# SANITY CHECK — forward pass
# ============================================================

print("=== Sanity check forward pass ===")

sample_records = meta[["path_h5", "epoch_idx", "label_idx", "subject_id"]].iloc[:8].to_dict("records")
ds_sample = EEGHypergraphDataset(
    sample_records, keep_idx, labelid2cluster,
    hyperedge_index, N_HYPER,
    mean=ds_train.mean, std=ds_train.std
)
batch_sample = next(iter(PyGDataLoader(ds_sample, batch_size=4, shuffle=False)))

print(f"Batch x:                {batch_sample.x.shape}      # (4×59, 384)")
print(f"Batch hyperedge_index:  {batch_sample.hyperedge_index.shape}")
print(f"  vertex range:         [0, {batch_sample.hyperedge_index[0].max().item()}]  (atteso: {4*N_CHANS-1})")
print(f"  hyperedge range:      [0, {batch_sample.hyperedge_index[1].max().item()}]  (atteso: {4*N_HYPER-1})")
print(f"Batch y:                {batch_sample.y}")

for name, cfg in HGNN_MODELS.items():
    m = cfg["cls"]().to(device)
    m.eval()
    with torch.no_grad():
        logits = m(batch_sample.to(device))
        print(f"  {name}: logits={logits.shape} ✓")
    m.to("cpu")

print("
Sanity check OK")

In [ ]:
# ============================================================
# FUNZIONE DI TRAINING (standard, senza adversarial)
# Identica struttura a EEG_09 train_gat — adattata per hypergraph
# ============================================================

def train_hgnn(model, ds_train, ds_val, save_path, tb_dir,
               n_epochs=MAX_EPOCHS, patience=PATIENCE,
               lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE,
               class_weights=None, resume=RESUME):
    """
    Training con early stopping su val_bacc.
    Class weights per gestire sbilanciamento concr4.
    CosineAnnealingLR + gradient clipping.
    """
    if resume and Path(save_path).exists():
        print(f"  Resume: {save_path} già esistente, skip training.")
        model.load_state_dict(torch.load(save_path, map_location="cpu"))
        return model, {}

    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    writer    = SummaryWriter(log_dir=str(tb_dir))

    loader_tr = PyGDataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=0)
    loader_va = PyGDataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=0)

    best_val_bacc, patience_cnt = -1.0, 0
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    history    = defaultdict(list)
    t0 = time.time()

    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────
        model.train()
        loss_sum, correct, n_tot = 0.0, 0, 0
        for data in tqdm(loader_tr, desc=f"Epoch {epoch+1}/{n_epochs}", leave=False):
            data = data.to(device)
            optimizer.zero_grad()
            logits = model(data)
            loss   = criterion(logits, data.y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item() * len(data.y)
            correct  += (logits.argmax(1) == data.y).sum().item()
            n_tot    += len(data.y)
        scheduler.step()

        # ── Validation ─────────────────────────────────────
        model.eval()
        ys_v, ps_v = [], []
        with torch.no_grad():
            for data in loader_va:
                data = data.to(device)
                ps_v.extend(model(data).argmax(1).cpu().tolist())
                ys_v.extend(data.y.cpu().tolist())

        val_acc  = accuracy_score(ys_v, ps_v)
        val_bacc = balanced_accuracy_score(ys_v, ps_v)
        tr_acc   = correct / n_tot if n_tot > 0 else 0.0
        tr_loss  = loss_sum / n_tot if n_tot > 0 else 0.0

        history["val_acc"].append(val_acc)
        history["val_bacc"].append(val_bacc)
        history["train_acc"].append(tr_acc)
        history["train_loss"].append(tr_loss)

        writer.add_scalar("val/acc",    val_acc,  epoch)
        writer.add_scalar("val/bacc",   val_bacc, epoch)
        writer.add_scalar("train/acc",  tr_acc,   epoch)
        writer.add_scalar("train/loss", tr_loss,  epoch)

        if val_bacc > best_val_bacc:
            best_val_bacc = val_bacc
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
        else:
            patience_cnt += 1

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f} "                  f"val_bacc={val_bacc:.3f} (best={best_val_bacc:.3f}) patience={patience_cnt}")

        if patience_cnt >= patience:
            print(f"  Early stop a epoch {epoch+1}")
            break

    elapsed = int(time.time() - t0)
    model.load_state_dict(best_state)
    model.to("cpu")
    torch.save(best_state, save_path)
    writer.close()
    print(f"  Terminato: best val_bacc={best_val_bacc:.4f} | tempo={elapsed}s")
    return model, history


def evaluate_model(model, ds_test, batch_size=BATCH_SIZE):
    """Valuta sul test set, restituisce acc, bacc, predizioni e label vere."""
    model = model.to(device)
    model.eval()
    loader = PyGDataLoader(ds_test, batch_size=batch_size, shuffle=False, num_workers=0)
    ys, ps = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            ps.extend(model(data).argmax(1).cpu().tolist())
            ys.extend(data.y.cpu().tolist())
    model.to("cpu")
    return (
        accuracy_score(ys, ps),
        balanced_accuracy_score(ys, ps),
        ys, ps
    )


print("Funzioni training pronte")

In [ ]:
# ============================================================
# TRAINING — tutti i modelli
# ============================================================

tb_root  = project_root / "runs" / "eeg10"
ckpt_dir = project_root / "data" / "interim" / "checkpoints_eeg10"
ckpt_dir.mkdir(parents=True, exist_ok=True)

results = []

for model_name, cfg in HGNN_MODELS.items():
    print(f"
{'='*55}")
    print(f"  Training: {model_name}")
    print(f"  Descrizione: {cfg['desc']}")
    print(f"{'='*55}")

    model_inst = cfg["cls"]()
    save_path  = ckpt_dir / f"{model_name}.pt"
    tb_dir     = tb_root  / model_name

    trained_model, history = train_hgnn(
        model       = model_inst,
        ds_train    = ds_train,
        ds_val      = ds_val,
        save_path   = save_path,
        tb_dir      = tb_dir,
        class_weights = class_weights,
        resume      = RESUME,
    )

    # Valutazione val set per best_val_bacc
    val_acc, val_bacc, _, _ = evaluate_model(trained_model, ds_val)
    test_acc, test_bacc, _, _ = evaluate_model(trained_model, ds_test)
    n_epochs_done = len(history.get("val_bacc", [1]))
    best_val_bacc = max(history.get("val_bacc", [val_bacc]))

    print(f"  val_bacc={val_bacc:.4f}  test_bacc={test_bacc:.4f}")

    results.append({
        "model":        model_name,
        "val_acc":      val_acc,
        "val_bacc":     val_bacc,
        "test_acc":     test_acc,
        "test_bacc":    test_bacc,
        "best_val_bacc":best_val_bacc,
        "epochs":       n_epochs_done,
    })

print("
Training completato per tutti i modelli.")

In [ ]:
# ============================================================
# RISULTATI + SALVATAGGIO CSV
# ============================================================

df_results = pd.DataFrame(results)
df_results.to_csv(RESULTS_CSV, index=False)
print(f"Risultati salvati in: {RESULTS_CSV}")
print()
print(df_results[["model", "val_bacc", "test_bacc", "epochs"]].to_string(index=False))

# Confronto con EEG_09
eeg09_csv = project_root / "data" / "interim" / f"eeg09_gat_{N_CLASSES}_results.csv"
if eeg09_csv.exists():
    df09 = pd.read_csv(eeg09_csv)
    print()
    print("=== Confronto EEG_09 (GAT) vs EEG_10 (HGNN) ===")
    chance = 1.0 / N_CLASSES
    print(f"Chance level: {chance:.4f} ({100*chance:.1f}%)")
    print()
    for _, row in df09.iterrows():
        print(f"  EEG_09 {row['model']:<20s}: test_bacc={row['test_bacc']:.4f}")
    for _, row in df_results.iterrows():
        delta = row["test_bacc"] - chance
        print(f"  EEG_10 {row['model']:<20s}: test_bacc={row['test_bacc']:.4f}  (+{delta:.4f} vs chance)")

In [ ]:
# ============================================================
# CONFUSION MATRICES (test set, normalizzate per riga)
# ============================================================

fig, axes = plt.subplots(1, len(HGNN_MODELS), figsize=(6 * len(HGNN_MODELS), 5))
if len(HGNN_MODELS) == 1:
    axes = [axes]

for ax, (model_name, cfg) in zip(axes, HGNN_MODELS.items()):
    model_inst = cfg["cls"]()
    ckpt = ckpt_dir / f"{model_name}.pt"
    if ckpt.exists():
        model_inst.load_state_dict(torch.load(ckpt, map_location="cpu"))

    _, test_bacc, ys, ps = evaluate_model(model_inst, ds_test)
    cm = confusion_matrix(ys, ps, normalize="true", labels=list(range(N_CLASSES)))

    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=cluster_names, yticklabels=cluster_names,
                ax=ax, vmin=0, vmax=1)
    ax.set_title(f"{model_name}
test_bacc={test_bacc:.3f}", fontsize=11)
    ax.set_xlabel("Predetto")
    ax.set_ylabel("Vero")

plt.tight_layout()
fig_path = project_root / "figures" / f"eeg10_confusion_matrices_{N_CLASSES}cls.png"
fig_path.parent.mkdir(exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")